# Scraper un site avec Scrapy

Cette démo écrit un *spider* Scrapy, la classe qui dit quelles pages visiter et quoi en extraire, puis le lance sans créer de projet. La cellule suivante installe Scrapy.

In [1]:
!pip install scrapy

  Using cached aiohttp-3.14.5-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached backports_zstd-1.8.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (6.8 kB)
  Using cached brotli-1.2.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.1 kB)
  Using cached cryptography-50.0.2-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (4.4 kB)
  Using cached cssselect-1.6.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached itemloaders-1.5.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached lxml-6.1.3-cp313-cp313-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (3.3 kB)
  Using cached parsel-1.12.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached protego-0.7.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached PyDispatcher-2.0.7-py3-none-any.whl.metadata (2.4 kB)
  Using cached pyopenssl-26.4.0-py3-

Le spider parcourt `quotes.toscrape.com`, un site d'entraînement au scraping, à partir de la page des citations humoristiques. Scrapy télécharge chaque page de `start_urls` et passe sa réponse à `parse`, qui y choisit les éléments par des sélecteurs CSS (`::text` en prend le texte, `get()` le premier résultat) : pour chaque citation (`div.quote`), il renvoie un dictionnaire de son auteur et de son texte. Enfin, `reponse.follow` crée la requête de la page suivante à partir du lien relatif du bouton « Next », et Scrapy appelle de nouveau `parse` sur sa réponse, jusqu'à la dernière page, qui n'a pas ce bouton.

In [2]:
%%writefile citations_spider.py
import scrapy


class CitationsSpider(scrapy.Spider):
    name = "citations"
    start_urls = [
        "https://quotes.toscrape.com/tag/humor/",
    ]

    def parse(self, reponse):
        for citation in reponse.css("div.quote"):
            yield {
                "auteur": citation.css("small.author::text").get(),
                "texte": citation.css("span.text::text").get(),
            }

        page_suivante = reponse.css('li.next a::attr("href")').get()
        if page_suivante is not None:
            yield reponse.follow(page_suivante, self.parse)

Writing citations_spider.py


`scrapy runspider` exécute le spider sans créer de projet Scrapy, et `-O` écrase `citations.jsonl` à chaque exécution (`-o` ajouterait à la fin du fichier).

**Politesse :** hors d'un projet, Scrapy ignore `robots.txt` (`ROBOTSTXT_OBEY` vaut `False`) et n'espace pas ses requêtes (`DOWNLOAD_DELAY` vaut `0`). Le site d'entraînement `quotes.toscrape.com` est fait pour être parcouru ; sur un vrai site, demandez ces deux réglages dans l'attribut `custom_settings` du spider :

```python
custom_settings = {"ROBOTSTXT_OBEY": True, "DOWNLOAD_DELAY": 1}
```

In [3]:
!scrapy runspider citations_spider.py -O citations.jsonl

2026-11-11 22:45:15 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-11-11 22:45:15 [asyncio] DEBUG: Using selector: EpollSelector
2026-11-11 22:45:15 [scrapy.addons] INFO: Enabled addons:
[]
2026-11-11 22:45:15 [scrapy.utils.log] INFO: Scrapy 2.20.0 started (bot: scrapybot)
2026-11-11 22:45:15 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.3',
 'libxml2': '2.14.6',
 'cssselect': '1.6.0',
 'parsel': '1.12.1',
 'w3lib': '2.6.0',
 'Twisted': '26.10.0',
 'Python': '3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ]',
 'pyOpenSSL': '26.4.0 (OpenSSL 4.0.3 29 Sep 2026)',
 'cryptography': '50.0.2',
 'Platform': 'Linux-6.12.116+deb13-amd64-x86_64-with-glibc2.41'}
2026-11-11 22:45:15 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-11-11 22:45:15 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-11-11 22:45:15 [scrapy.extensions.telnet] INFO: Telnet Password: 26162693bfcec092
2026-11

In [4]:
!cat citations.jsonl

{"auteur": "Jane Austen", "texte": "\u201cThe person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.\u201d"}
{"auteur": "Steve Martin", "texte": "\u201cA day without sunshine is like, you know, night.\u201d"}
{"auteur": "Garrison Keillor", "texte": "\u201cAnyone who thinks sitting in church can make you a Christian must also think that sitting in a garage can make you a car.\u201d"}
{"auteur": "Jim Henson", "texte": "\u201cBeauty is in the eye of the beholder and it may be necessary from time to time to give a stupid or misinformed beholder a black eye.\u201d"}
{"auteur": "Charles M. Schulz", "texte": "\u201cAll you need is love. But a little chocolate now and then doesn't hurt.\u201d"}
{"auteur": "Suzanne Collins", "texte": "\u201cRemember, we're madly in love, so it's all right to kiss me anytime you feel like it.\u201d"}
{"auteur": "Charles Bukowski", "texte": "\u201cSome people never go crazy. What truly horrible lives they must lead.\u20